# The DAG and Query Execution Plans: Parsed, Analyzed, Optimized, and Physical

**Module**: 04_execution_engine  
**Notebook**: 02_dag_and_execution_plans.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Catalyst converts raw code into the 4 plan stages: Parsed Logical Plan, Analyzed Logical Plan, Optimized Logical Plan, and Physical Plan.
- **Why does this topic exist?**: Reading `df.explain()` is the #1 forensic skill for Senior Data Engineers to diagnose query performance.
- **What problem does it solve?**: Queries fail or run slowly because of unnoticed Cartesian joins, missing predicate pushdown, or unnecessary shuffles.
- **Where is it used in real-world Data Engineering?**: Optimizing multi-hour enterprise ETL jobs by analyzing explain plan trees.

---

## 2. The 4 Stages of Catalyst Query Planning

```
User Code / SQL
       |
       v
1. PARSED LOGICAL PLAN (Unresolved: verifies syntax; column names unresolved)
       |
       v (Catalog Resolution: checks table schemas, column types)
2. ANALYZED LOGICAL PLAN (Resolved: all columns and functions verified)
       |
       v (Rule-Based Optimization: Predicate Pushdown, Column Pruning, Constant Folding)
3. OPTIMIZED LOGICAL PLAN (Minimal algebraic operations)
       |
       v (Cost-Based Model: Join Strategy Selection, Algorithm Choice)
4. PHYSICAL PLAN (Concrete executable operators: FileScan, HashAggregate, SortMergeJoin)
       |
       v (Tungsten: WholeStageCodegen)
  Java Bytecode Execution
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Execution_Plans_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Inspecting All 4 Plan Stages with `explain(True)`

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/products.csv")

query_df = (
    orders
    .filter("quantity > 2")
    .join(products, on="product_id")
    .filter("category = 'Electronics'")
    .select("order_id", "product_name", "quantity", "unit_price")
)

# explain(True) prints Parsed, Analyzed, Optimized, and Physical plans
query_df.explain(True)

## 4. Modern Formatted Execution Plan with `explain('formatted')`

In [ ]:
print("=== Formatted Plan Output ===")
query_df.explain("formatted")

---

## Experiment: Predicate Pushdown Observation in Physical Plan

- **1. What are we testing?**: Verify in the Physical Plan that'category = Electronics'is pushed down directly into the FileScan node.
- **2. What do we expect?**: PushedFilters in FileScan contains StringEqualTo(category, Electronics).

In [ ]:
print("Check FileScan PushedFilters:")
query_df.explain("cost")

**3. What happened?**: FileScan showed PushedFilters directly reading matching rows.  
**4. Why did it happen?**: Catalyst moved the filter condition past the join directly into the table reader.  
**5. Practical Engineering Lesson**: Always verify that your filters appear under PushedFilters in physical plans to prevent scanning unneeded data.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify WholeStageCodegen in Plan
**Problem Statement**: Run explain() and identify which physical plan operators are prefixed with an asterisk (*), indicating WholeStageCodegen.  
**Input / Context**: query_df

In [ ]:
# Exercise 1: Identify codegen operators

### Exercise 2 (Intermediate): Detect Exchange (Shuffle) Operators
**Problem Statement**: Write a query that groups and sorts data, and count how many Exchange nodes appear in explain().  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 2: Count Exchange nodes

### Exercise 3 (Advanced): Analyze Join Strategy in Physical Plan
**Problem Statement**: Inspect query_df and determine whether Spark selected BroadcastHashJoin or SortMergeJoin.  
**Input / Context**: query_df

In [ ]:
# Exercise 3: Join strategy identification

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify WholeStageCodegen in Plan):
```python
print("Operators with * run inside WholeStageCodegen:")
for line in query_df._jdf.queryExecution().executedPlan().toString().split("\n"):
    if"*"in line:
        print("->", line.strip())
```

#### Solution 2 (Detect Exchange (Shuffle) Operators):
```python
p = orders.groupBy("status").count().orderBy("count")
plan_str = p._jdf.queryExecution().executedPlan().toString()
print("Exchange count:", plan_str.count("Exchange"))
```

#### Solution 3 (Analyze Join Strategy in Physical Plan):
```python
plan = query_df._jdf.queryExecution().executedPlan().toString()
if"BroadcastHashJoin"in plan:
    print("Selected Join: BroadcastHashJoinExec")
elif"SortMergeJoin"in plan:
    print("Selected Join: SortMergeJoinExec")
else:
    print("Other Join Strategy Selected")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **4 Plan Levels**: Parsed -> Analyzed -> Optimized -> Physical.
- **explain(True)**: The universal tool for inspecting Catalyst decision-making.
- **PushedFilters**: Indicates storage-level filtering (predicate pushdown).

### Essential Technical Terms
- `Parsed Plan`: Unresolved syntax tree.
- `Analyzed Plan`: Resolved against catalog metadata.
- `Physical Plan`: Executable algorithm graph.

### Common Engineering Mistakes
- **Ignoring Exchange nodes**: Missing that your query is doing 5 unnecessary network shuffles.
- **Assuming queries execute imperatively**: Catalyst reorders filters and joins for optimal execution.

### Interview Questions on This Topic
- **Q: What is the difference between Logical Plan and Physical Plan?**
  - *A*: The Logical Plan describes WHAT data to compute; the Physical Plan describes HOW to compute it (which concrete algorithms and join strategies to execute).
- **Q: What does an asterisk (*) next to an operator in explain() mean?**
  - *A*: It signifies that the operator participates in Whole-Stage Code Generation (Tungsten).

### What to Study Next
Open 04_execution_engine/03_dag_scheduler.ipynb to dissect the DAGScheduler.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')